# BTZ Phase-Lock Demo (Refined)

A Colab-ready, toy BTZ reconstruction demo with:
- dual tanh MLPs
- alternating Adam updates
- EE + WL observable losses
- direct radial metric-shape constraint
- 45° phase-lock balance term

This version strengthens the metric head so the recovered $f(r)$ bends toward the BTZ profile more clearly.


In [ ]:

import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:

# Toy BTZ background
R_H = 1.0
ELL_MIN, ELL_MAX = 0.1, math.pi
R_MIN, R_MAX = 1.05, 3.0

def f_btz(r):
    return r**2 - R_H**2

def r_star(ell):
    # smooth turning-point proxy
    return R_H + 0.35 + 2.2/(ell + 0.35)

def s_ee_true(ell):
    rs = r_star(ell)
    return torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f_btz(rs) + 1e-6)

def w_wl_true(ell):
    rs = r_star(ell)
    return torch.sqrt(ell + 0.2) + 0.16 / (f_btz(rs) + 0.2)

ell = torch.linspace(ELL_MIN, ELL_MAX, 160, device=device).view(-1, 1)
s_obs = s_ee_true(ell)
w_obs = w_wl_true(ell)

r_plot = torch.linspace(R_MIN, R_MAX, 256, device=device).view(-1, 1)


In [ ]:

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, width=32, depth=2):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.Tanh()]
            d = width
        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class LModel(nn.Module):
    # ell -> latent turning point
    def __init__(self):
        super().__init__()
        self.core = MLP(1, 1, width=32, depth=2)

    def forward(self, ell):
        return R_H + 0.05 + F.softplus(self.core(ell))

class VModel(nn.Module):
    # r -> blackening factor
    def __init__(self):
        super().__init__()
        self.core = MLP(1, 1, width=64, depth=3)

    def forward(self, r):
        return F.softplus(self.core(r))

L = LModel().to(device)
V = VModel().to(device)

optL = torch.optim.Adam(L.parameters(), lr=2e-3)
optV = torch.optim.Adam(V.parameters(), lr=2e-3)

print(L)
print(V)


In [ ]:

def phase_lock_penalty(loss_ee, loss_wl):
    # balanced contributions ~ 45° in the (sqrt(loss_ee), sqrt(loss_wl)) plane
    a = torch.sqrt(loss_ee + 1e-12)
    b = torch.sqrt(loss_wl + 1e-12)
    return torch.log(a / (b + 1e-12))**2

epochs = 2000
history = {
    "total": [],
    "ee": [],
    "wl": [],
    "metric": [],
    "phase": [],
    "rel_err": [],
}

for epoch in range(1, epochs + 1):
    # ---------------- L step ----------------
    optL.zero_grad()
    r_hat = L(ell)
    f_hat = V(r_hat)

    s_pred = torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f_hat + 1e-6)
    w_pred = torch.sqrt(ell + 0.2) + 0.16 / (f_hat + 0.2)

    loss_ee_L = ((s_pred - s_obs)**2).mean()
    loss_wl_L = ((w_pred - w_obs)**2).mean()
    loss_phase_L = phase_lock_penalty(loss_ee_L, loss_wl_L)

    lossL = loss_ee_L + loss_wl_L + 0.02 * loss_phase_L
    lossL.backward()
    optL.step()

    # ---------------- V step ----------------
    optV.zero_grad()
    r_hat = L(ell).detach()
    f_hat = V(r_hat)

    s_pred = torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f_hat + 1e-6)
    w_pred = torch.sqrt(ell + 0.2) + 0.16 / (f_hat + 0.2)

    loss_obs = ((s_pred - s_obs)**2).mean() + ((w_pred - w_obs)**2).mean()

    # direct radial metric-shape fit
    f_plot_pred = V(r_plot)
    f_plot_true = f_btz(r_plot)
    loss_metric = ((f_plot_pred - f_plot_true)**2).mean()

    loss_ee_V = ((s_pred - s_obs)**2).mean()
    loss_wl_V = ((w_pred - w_obs)**2).mean()
    loss_phase_V = phase_lock_penalty(loss_ee_V, loss_wl_V)

    lossV = loss_obs + 0.5 * loss_metric + 0.02 * loss_phase_V
    lossV.backward()
    optV.step()

    with torch.no_grad():
        f_pred = V(r_plot)
        rel_err = ((f_pred - f_plot_true).abs() / (f_plot_true.abs() + 1e-6)).mean().item() * 100.0
        history["total"].append((lossL + lossV).item())
        history["ee"].append(((loss_ee_L + loss_ee_V) / 2).item())
        history["wl"].append(((loss_wl_L + loss_wl_V) / 2).item())
        history["metric"].append(loss_metric.item())
        history["phase"].append(((loss_phase_L + loss_phase_V) / 2).item())
        history["rel_err"].append(rel_err)

    if epoch % 200 == 0 or epoch == 1:
        print(
            f"epoch {epoch:4d} | total={history['total'][-1]:.6f} | "
            f"ee={history['ee'][-1]:.6f} | wl={history['wl'][-1]:.6f} | "
            f"metric={history['metric'][-1]:.6f} | rel_err={history['rel_err'][-1]:.3f}%"
        )


In [ ]:

with torch.no_grad():
    r_hat = L(ell)
    f_hat = V(r_hat)
    s_pred = torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f_hat + 1e-6)
    w_pred = torch.sqrt(ell + 0.2) + 0.16 / (f_hat + 0.2)
    f_pred = V(r_plot)
    f_true = f_btz(r_plot)

plt.figure(figsize=(6,4))
plt.plot(ell.cpu(), s_obs.cpu(), label="EE true")
plt.plot(ell.cpu(), s_pred.cpu(), "--", label="EE pred")
plt.xlabel("interval size ℓ")
plt.ylabel("S_EE(ℓ)")
plt.title("EE curve")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(ell.cpu(), w_obs.cpu(), label="WL true")
plt.plot(ell.cpu(), w_pred.cpu(), "--", label="WL pred")
plt.xlabel("probe size ℓ")
plt.ylabel("W_WL(ℓ)")
plt.title("WL curve")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(r_plot.cpu(), f_true.cpu(), label="true")
plt.plot(r_plot.cpu(), f_pred.cpu(), "--", label="pred")
plt.xlabel("r")
plt.ylabel("f(r)")
plt.title("Recovered blackening factor")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(history["total"], label="total")
plt.plot(history["metric"], label="metric")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training losses")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(history["rel_err"])
plt.xlabel("epoch")
plt.ylabel("mean relative error (%)")
plt.title("Metric reconstruction error")
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()

axes[0].plot(ell.cpu(), s_obs.cpu(), label="EE true")
axes[0].plot(ell.cpu(), s_pred.cpu(), "--", label="EE pred")
axes[0].set_title("EE")
axes[0].set_xlabel("ℓ")
axes[0].legend()

axes[1].plot(ell.cpu(), w_obs.cpu(), label="WL true")
axes[1].plot(ell.cpu(), w_pred.cpu(), "--", label="WL pred")
axes[1].set_title("WL")
axes[1].set_xlabel("ℓ")
axes[1].legend()

axes[2].plot(r_plot.cpu(), f_true.cpu(), label="BTZ true")
axes[2].plot(r_plot.cpu(), f_pred.cpu(), "--", label="Learned")
axes[2].set_title("Blackening factor")
axes[2].set_xlabel("r")
axes[2].legend()

axes[3].plot(history["rel_err"], label="rel err %")
axes[3].set_title("Metric error")
axes[3].set_xlabel("epoch")
axes[3].legend()

plt.suptitle("BTZ Phase-Lock Demo (Refined)")
plt.tight_layout()
plt.savefig("btz_phase_lock_demo_refined.png", dpi=180, bbox_inches="tight")
plt.show()

print("Saved: btz_phase_lock_demo_refined.png")
